# braga2025annotation

> Semi-automatic annotation process of AIRCloud. Labels and point cloud are saved in **original point cloud order** using idx_map (no reordering).

In [ ]:
#| eval: false
from torchvision.transforms import v2
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from colorcloud.behley2019iccv import SemanticKITTIDataset, SphericalProjection
from colorcloud.behley2019iccv import ProjectionTransform, ProjectionToTensorTransform
import os
import yaml
import shutil
import zipfile

## Introduction

This module aims to simplify and accelerate the process of annotating 3D point clouds through a semi-automatic approach. The methodology adopted is based on using a semantic segmentation model as the initial tool for automatic annotation. This model performs inference on the point cloud, generating preliminary labels for the various elements present in the scene.

Subsequently, a human verifier reviews these annotations using a specific annotation tool, correcting any errors made by the model. After this stage, the validated annotations are incorporated into the dataset, expanding it and enabling the training of a new segmentation model, thus progressively improving the quality of the annotations generated in each iteration.

It is important to emphasize that the human verifier corrects only the new annotations produced by the model. Previously reviewed annotations are marked with a distinct label during the verification process, helping to identify new points.

## Setup

The first step is to define the folder paths.

- `DATASET_PATH`: Path containing the dataset
- `OUTPUT_PATH`: Location where the ZIP file with the labels will be saved
- `SEQUENCE` : Selected sequence

::: {.callout-important}
The script currently expects only one dataset sequence at a time.
:::

In [ ]:
#| eval: false
DATASET_PATH = "/workspace/data/AIRCloud/"
OUTPUT_PATH = "/workspace/data/results/AIRCloud/"
SEQUENCE = "08"

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## Merge Dataset Labels

::: {.callout-important}
If this is the first iteration of the dataset in the semi-automatic annotation process, skip this step.
:::

After completing the corrections using the annotation tool, the final dataset must be created by merging the previously verified annotations with the newly generated ones

The first step is to add the compressed folder containing the labels to `/workspace/data/` and set its path in `ZIP_PATH`. Also, define the `CORRECTED_DATA_PATH`, the path from which the labels will be extracted.

In [ ]:
#| eval: false
ZIP_PATH = "/workspace/data/corrected/AIRCloud/08.zip"
CORRECTED_DATA_PATH = '/workspace/merged/08'
os.makedirs(CORRECTED_DATA_PATH, exist_ok=True)

def unzip_file(zip_file_path, destination):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(destination)
        print(f"Files extracted to: {destination}")

os.makedirs(CORRECTED_DATA_PATH, exist_ok=True)
unzip_file(ZIP_PATH, CORRECTED_DATA_PATH)

The next cell loads label files aligned with their corresponding Velodyne point clouds, filtering out invalid points (NaNs).
This function is used to load both the newly corrected labels and the original dataset labels, ensuring that label arrays remain consistent with the filtered point cloud data.

- `labels_path`: path to the directory containing label files
- `label_list`: sorted list of label filenames
- `velodyne_path`: path to the directory containing Velodyne point cloud files
- `velodyne_list`: sorted list of Velodyne filenames


In [ ]:
#| eval: false
def load_labels(labels_path, label_list, velodyne_path, velodyne_list):
    labels = []
    for index in (range(len(label_list))):
        # Load velodyne 
        velodyne_file = velodyne_list[index]
        velodyne_file_path = os.path.join(velodyne_path, velodyne_file)
        with open(velodyne_file_path, 'rb') as f:
            frame = np.fromfile(f, dtype=np.float32).reshape(-1, 4)
        
        # Load label
        label_path = os.path.join(labels_path, label_list[index])
        with open(label_path, 'rb') as f:
            label = np.fromfile(f, dtype=np.uint32)
            label = label & 0xFFFF
        
        # Remove nans from velodyne and filter labels accordingly
        nan_mask = ~np.isnan(frame).any(axis=1)
        frame = frame[nan_mask]
        label = label[nan_mask]
        
        labels.append(label)

    return labels

new_labels_path = os.path.join(CORRECTED_DATA_PATH, SEQUENCE, "labels")
new_labels_list = sorted(os.listdir(new_labels_path))
new_velodyne_path = os.path.join(CORRECTED_DATA_PATH, SEQUENCE, "velodyne")
new_velodyne_list = sorted(os.listdir(new_velodyne_path))
new_labels = load_labels(new_labels_path, new_labels_list, new_velodyne_path, new_velodyne_list)

old_labels_path = os.path.join(DATASET_PATH, "data_odometry_labels", "dataset", "sequences", SEQUENCE, "labels")
old_labels_list = sorted(os.listdir(old_labels_path))
old_velodyne_path = os.path.join(DATASET_PATH, "data_odometry_velodyne", "dataset", "sequences", SEQUENCE, "velodyne")
old_velodyne_list = sorted(os.listdir(old_velodyne_path))
old_labels = load_labels(old_labels_path, old_labels_list, old_velodyne_path, old_velodyne_list)

After that, for each element in the dataset, we analyze the value present in `new_labels`, which leads to the following possible scenarios:

- If the value is `0`, no label was kept for that position, so the value from `old_labels` is used.
- If the value is `99`, the label at that position was previously annotated and verified, so the value from `old_labels` is used.
- If neither of the above conditions is met, the new value from `new_labels` is used.

In [ ]:
#| eval: false
mapped_labels = []
for old, new in zip(old_labels, new_labels):
    mask = (new != 99) & (new != 0)
    mapped = np.where(mask, new, old)
    mapped_labels.append(mapped)

Finally, each array in `mapped_labels` is converted to `uint32` and saved using the original filenames from `new_labels_list`, ensuring compatibility with the dataset structure.

The labels are temporarily stored in a folder created under `RESULT_PATH`. Once all files are written, this folder is compressed into a `.zip` archive and then removed.

The resulting archive contains the final set of labels ready for use. Since the files preserve the original order and naming of the point cloud frames, updating the dataset simply requires replacing the previous label files with the newly generated ones.

In [ ]:
#| eval: false
RESULT_PATH = "/workspace/converted_labels"
TEMP_PATH = os.path.join(RESULT_PATH, "temp_labels")
ZIP_NAME = "10.1_teste"

os.makedirs(TEMP_PATH, exist_ok=True)

for filename, arr in zip(new_labels_list, mapped_labels):
    final_labels_uint32 = arr.astype(np.uint32)
    out_path = os.path.join(TEMP_PATH, filename)
    with open(out_path, "wb") as f:
        final_labels_uint32.tofile(f)

zip_filename = os.path.join(RESULT_PATH, ZIP_NAME)
shutil.make_archive(zip_filename, "zip", TEMP_PATH)

shutil.rmtree(TEMP_PATH)

print(f"Labels ready for download at {zip_filename}.zip")

## Inference

In this example, the network used will be **RIUNet**, which takes as input point clouds previously processed into range images. Since **AIRCloud** was built with a structure very similar to **SemanticKITTI**, we can use the `00_behley2019iccv` module to process and load the dataset, simply by adjusting the projection parameters and passing the `aircloud` flag to the class. For more information on how to load the dataset, refer to the `00_behley2019iccv` notebook.

::: {.callout-important}
The `SemanticKITTIDataset` class from the `00_behley2019iccv` module, although it fully supports loading **AIRCloud**, does not yet provide ways to specify which  `.yaml` file to use or which sequences to load. To work around this, it is important to modify the `.yaml` in the Splits section: set the desired sequence under the `valid` key and use the parameter `split="valid"`. This way, only the target sequence will be loaded.
:::

First, we define the projection, the transformation pipeline and the dataset.

In [ ]:
#| eval: false
proj = SphericalProjection(fov_up_deg=15., fov_down_deg=-15., W=1824, H=16)
tfms = v2.Compose([
    ProjectionTransform(proj),
    ProjectionToTensorTransform(),
])

ds = SemanticKITTIDataset(
    data_path=DATASET_PATH, 
    split='valid', 
    transform=tfms, 
    aircloud=True
)

print("Size of the dataset:",len(ds))

In [ ]:
#| eval: false
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Next, the data loader definition and model loading:
- `ckpt_path`: Path to the model checkpoint

In [ ]:
#| eval: false
batch_size=8
data_loader = DataLoader(
    ds, 
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

ckpt_path = "/workspace/models/AIRCloud_V1.pt"
model = torch.load(ckpt_path)

Then, inference is performed on the dataset, and the labels predicted by the model for each frame are stored in a NumPy array.

In [ ]:
#| eval: false
def get_prediction(single_pred):
    np_pred = single_pred[0].detach().cpu().numpy()
    pred_labels = np.argmax(np_pred, axis=0)  
    return pred_labels

def inference(model, data_loader, device):
    model.eval()
    num_batches = 0
    pred_np = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Inference", leave=True):
            num_batches += 1

            test_item = {key: value.to(device) for key, value in batch.items()}
            img = test_item['frame']
            label = test_item['label']
            mask = test_item['mask']

            label[~mask] = 0
            pred = model(img) 
            for i in range(pred.shape[0]):
                single_pred = pred[i].unsqueeze(0)  # Shape: (1, C, H, W)
                pred_np.append(get_prediction(single_pred))
                
    return pred_np

pred_np = inference(model, data_loader, device)

Next, an auxiliary array is created based on the dataset, aiming to replace all non-zero points (i.e., previously annotated points) with a value that distinguishes them from others during the verification process.

::: {.callout-note}
The chosen value was 99, which in the `semantic-kitti.yaml` file corresponds to the other-object class, a class that is not used in AIRCloud.
:::

In [ ]:
#| eval: false
aux_np = []
for frame in ds:
    step_1 = np.where(frame["label"] == -1, 0, frame["label"])
    step_2 = np.where(step_1 != 0, 99, step_1)
    aux_np.append(step_2)
np.unique(aux_np)

Finally, the `filtered_prediction` array is created. For each point equal to zero in the auxiliary array (i.e., points without prior annotation), the value is replaced by the model’s prediction. When the point is non-zero (specifically 99), the value is preserved.

In this way, the final array contains the verified annotation value (marked as other-object) if it exists, and otherwise, it takes the predicted value.

In [ ]:
#| eval: false
from typing import Any

filtered_prediction = [
    np.where(a == 0, p, a)   
    for a, p in zip(aux_np, pred_np)
]
np.unique(filtered_prediction[0])

## Converting the results

After the inference step and the composition of the dataset to be corrected, the next step is to project the results back into 3D so they can be downloaded and the labels corrected using the annotation tool.

The following cell reloads the dataset with `return_original=True` to preserve the original point cloud structure. 

Unlike the inference dataset (which uses `ProjectionToTensorTransform()` for PyTorch tensors), this loading keeps NumPy arrays and includes essential metadata: 
- `frame_original`: original point cloud copy
- `idx_map`: 2D array mapping projection pixels to original point indices
- `original_len`: original point count

These fields are required to map 2D predictions back to 3D points in their original order.

In [ ]:
#| eval: false
proj = SphericalProjection(fov_up_deg=15., fov_down_deg=-15., W=1824, H=16)
tfms = v2.Compose([
    ProjectionTransform(proj, return_original=True),
])

ds = SemanticKITTIDataset(
    data_path=DATASET_PATH, 
    split='valid', 
    transform=tfms, 
    aircloud=True
)

print("Size of the dataset:",len(ds))

The code below creates output directories for point clouds (`velodyne/`) and labels (`labels/`) within the sequence folder. It defines two helper functions: 
* `fname_for()`: return the original label filename for each frame
* `save_scan()`: writes point cloud data as little-endian float32 binary files (`.bin`) and labels as uint32 binary files (`.label`), masking labels with `0xFFFF` to remove instance bits and keep only semantic labels in the lower 16 bits, compatible with SemanticKITTI format.

In [ ]:
#| eval: false
out_seq = os.path.join(OUTPUT_PATH, SEQUENCE)
velodyne_dir = os.path.join(out_seq, "velodyne")
labels_dir = os.path.join(out_seq, "labels")
os.makedirs(velodyne_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

labels_path_orig = os.path.join(ds.labels_path, SEQUENCE, "labels")
label_list_orig = sorted(os.listdir(labels_path_orig)) if os.path.isdir(labels_path_orig) else []

def fname_for(i):
    return os.path.splitext(label_list_orig[i])[0]

def save_scan(bin_path, label_path, pc_xyzr, labels_uint32):
    pc_xyzr.astype('<f4').tofile(bin_path)
    (labels_uint32.astype(np.uint32) & np.uint32(0xFFFF)).tofile(label_path)

After that, this cell converts 2D image predictions back to 3D point cloud labels using the `idx_map` to map each projection pixel to its original point index. For each frame, it initializes an output label array of size `original_len`, extracts valid pixel-to-point mappings from `idx_map`, assigns predicted labels to corresponding original point positions, and converts internal class IDs to SemanticKITTI standard IDs using `inv_map`. 

After ensuring `frame_original` and `output_label` have matching lengths, it saves both files and stores results. This preserves the original point cloud order throughout the conversion.

In [ ]:
#| eval: false
inv_map = {
    0: 0,     # unlabeled
    1: 10,    # car
    2: 30,    # person
    3: 40,    # road
    4: 70,    # vegetation
    5: 72,    # terrain
    99: 99    # other-object
}

results = {"frame": [], "label": []}

for i, (d, label_img) in enumerate(zip(ds, filtered_prediction)):
    idx_map = d["idx_map"]
    original_len = d["original_len"]
    frame_original = d["frame_original"]

    label_img = np.asarray(label_img)
    output_label = np.zeros(original_len, dtype=np.uint32)
    valid = idx_map >= 0
    valid_indices = idx_map[valid]
    valid_labels = label_img[valid]
    in_range = valid_indices < original_len
    output_label[valid_indices[in_range]] = valid_labels[in_range]

    map_func = np.vectorize(lambda x: inv_map.get(int(x), 0))
    output_label = map_func(output_label).astype(np.uint32)

    if frame_original.shape[0] != output_label.shape[0]:
        min_len = min(frame_original.shape[0], output_label.shape[0])
        frame_original = frame_original[:min_len]
        output_label = output_label[:min_len]

    results["frame"].append(frame_original)
    results["label"].append(output_label)

    base = fname_for(i)
    save_scan(
        os.path.join(velodyne_dir, f"{base}.bin"),
        os.path.join(labels_dir, f"{base}.label"),
        frame_original,
        output_label)

Finally, this cell packages the converted sequence directory (containing `velodyne/` and `labels/` subdirectories) into a ZIP archive at `{OUTPUT_PATH}/{SEQUENCE}.zip`. 

The archive contains the previous point cloud (without NaNs) and the new labels generated from the inference process. This allows opening the data in the point labeler to perform necessary corrections. Note that points previously annotated in earlier iterations are marked as the other-object class (displayed in cyan), which helps distinguish them from new predictions during the correction process.

After the correction, it is possible to return to the notebook and use the **Merged Dataset Labels** section to merge the old annotated labels with the new corrected labels, and obtain the final result of the semi-automatically annotated dataset.

In [ ]:
#| eval: false
zip_filename = os.path.join(OUTPUT_PATH, SEQUENCE)
shutil.make_archive(zip_filename, 'zip', out_seq)
print(f"ZIP generated at: {zip_filename}.zip")